In [ ]:
# Configure the repository root once so the same notebook works locally and on Kaggle.
import os
import sys
from pathlib import Path

_configured_root = os.environ.get("DATA_DIR")
if _configured_root:
    DATA_DIR = Path(_configured_root).expanduser().resolve()
else:
    DATA_DIR = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "README.md").is_file() and (p / "data").is_dir()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError("Set DATA_DIR to the repository root before running this notebook.")
os.environ["DATA_DIR"] = str(DATA_DIR)
os.chdir(DATA_DIR)
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))


In [ ]:
import time
import os
import json
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import sys

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")
print("[Info] PyTorch:", torch.__version__)
print("[Info] CUDA khả dụng:", torch.cuda.is_available())

sys.path.append(str(DATA_DIR / "src/eval")) 
from evaluator import evaluate

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

Sử dụng thiết bị: cuda
[Info] PyTorch: 2.11.0+cu128
[Info] CUDA khả dụng: True


In [3]:
import torch 
def topk_random_ties(sc, K, g):
    """sc: [B, N] điểm, các khóa bị loại có giá trị -inf. Trả về chỉ số top-K [B, K]."""
    perm = torch.argsort(torch.rand(sc.shape, generator=g), dim=1)   # hoán vị riêng từng hàng
    sp = torch.gather(sc, 1, perm)
    order = torch.sort(sp, dim=1, descending=True, stable=True).indices[:, :K]
    return torch.gather(perm, 1, order).numpy()

@torch.no_grad() 
def evaluate(scores_fn, target, mask, ks=(5, 10, 20), cand=None, bs=1024, tie_seed=0): 
    users = sorted(target); K = max(ks) 
    out = {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks} 
    g = torch.Generator().manual_seed(tie_seed) 
    disc = 1.0 / np.log2(np.arange(2, K + 2)) 
    
    for s in range(0, len(users), bs): 
        ub = users[s:s + bs]; sc = scores_fn(torch.tensor(ub)).float().cpu() 
        if cand is not None: sc[:, ~torch.as_tensor(cand)] = -float("inf") 
        for r, u in enumerate(ub): 
            if mask.get(u): sc[r, list(mask[u])] = -float("inf") 
        
        # Phá hòa ngẫu nhiên riêng biệt cho từng học viên
        top = topk_random_ties(sc, K, g)
        
        for r, u in enumerate(ub): 
            gt = target[u]; hit = np.isin(top[r], list(gt)).astype(float) 
            for k in ks: 
                h = hit[:k]; idcg = disc[:min(len(gt), k)].sum() 
                out[f"Recall@{k}"].append(h.sum() / len(gt)) 
                out[f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg) 
                out[f"HR@{k}"].append(float(h.sum() > 0)) 
    return {m: np.array(v) for m, v in out.items()} 

In [ ]:
# 1. Đọc dữ liệu phân chia cold-scenario
tr_df = pd.read_parquet(str(DATA_DIR / "data/splits/cold_scenario/train_cold.parquet"))
te_df = pd.read_parquet(str(DATA_DIR / "data/splits/cold_scenario/test_cold.parquet"))

# 2. Đọc danh sách khóa lạnh test
with open(str(DATA_DIR / "data/splits/cold_scenario/cold_items_test.json"), 'r') as f:
    cold_items_test = json.load(f)

# 3. Nạp ma trận embedding và tệp profile đầy đủ (chứa đúng danh sách course_id theo đúng thứ tự 3.769 khóa)
raw_embeddings = np.load(str(DATA_DIR / "data/course_llm/1_embeddings/raw_course_text_embeddings.npy"))
llm_embeddings = np.load(str(DATA_DIR / "data/course_llm/1_embeddings/llm_profile_embeddings.npy"))

with open(str(DATA_DIR / "data/course_llm/2_profiles_full_and_shards/full_3769_profiles.json"), 'r') as f:
    full_profiles = json.load(f)

# Xây dựng từ điển ánh xạ course_id chính xác dựa theo thứ tự dòng trong ma trận embedding (0 đến 3768)
id_to_index = {}
index_to_id = {}
for idx, profile in enumerate(full_profiles):
    # Lấy course_id từ key trong mỗi bản ghi profile
    # Kiểm tra cấu trúc key của profile mẫu
    cid = profile.get('course_id') #or profile.get('id')
    if cid:
        id_to_index[str(cid)] = idx
        index_to_id[idx] = str(cid)

num_total_courses = raw_embeddings.shape[0]
cand_bool_array = np.zeros(num_total_courses, dtype=bool)

matched_count = 0
for cid in cold_items_test:
    str_cid = str(cid)
    if str_cid in id_to_index:
        idx = id_to_index[str_cid]
        if 0 <= idx < num_total_courses:
            cand_bool_array[idx] = True
            matched_count += 1

cold_cand_tensor = torch.tensor(cand_bool_array, dtype=torch.bool)
print(f"Tổng số khóa trong không gian embedding: {num_total_courses}")
print(f"Số lượng khóa lạnh khớp thành công trong cand_tensor: {matched_count} / {len(cold_items_test)}")

# 4. Chuẩn hóa ánh xạ user_id sang integer index cho hàm evaluate()
raw_target_dict = te_df.groupby('user_id')['course_id'].apply(list).to_dict()
unique_users = sorted(raw_target_dict.keys())
user_to_int = {u: i for i, u in enumerate(unique_users)}
int_to_user = {i: u for i, u in enumerate(unique_users)}

target_dict = {
    user_to_int[u]: [id_to_index[str(cid)] for cid in gt if str(cid) in id_to_index] 
    for u, gt in raw_target_dict.items() if u in user_to_int
}

raw_mask_dict = tr_df.groupby('user_id')['course_id'].apply(list).to_dict()
mask_dict = {
    user_to_int[u]: [id_to_index[str(cid)] for cid in hist if str(cid) in id_to_index] 
    for u, hist in raw_mask_dict.items() if u in user_to_int
}

user_history = raw_mask_dict

Tổng số khóa trong không gian embedding: 3769
Số lượng khóa lạnh khớp thành công trong cand_tensor: 502 / 502


In [5]:
def build_user_embedding_matrix(embeddings_matrix, user_history_dict, c2i):
    """
    Cơ chế Content-KNN: 
    Biểu diễn học viên (u) = trung bình cộng vector embedding của tất cả 
    các khóa học mà học viên đó đã tương tác trong tập huấn luyện (train).
    """
    user_embs = {}
    for u, hist in user_history_dict.items():
        v_idxs = [c2i[str(cid)] for cid in hist if str(cid) in c2i]
        if len(v_idxs) > 0:
            u_emb = embeddings_matrix[v_idxs].mean(axis=0)
            norm = np.linalg.norm(u_emb)
            if norm > 0:
                u_emb = u_emb / norm # Chuẩn hóa L2 vector học viên
            user_embs[u] = u_emb
        else:
            user_embs[u] = np.zeros(embeddings_matrix.shape[1])
    return user_embs

In [6]:
cand_bool_array_b = np.ones(num_total_courses, dtype=bool)
cand_tensor_b = torch.tensor(cand_bool_array_b, dtype=torch.bool)

print(f"Số lượng khóa được phép gợi ý ở tập (b): {cand_tensor_b.sum().item()} / {num_total_courses}")

def evaluate_content_knn_model_on_b(embeddings_matrix, name="Model (Tập b)"):
    print(f"\n================ ĐANG CHẠY ĐÁNH GIÁ TẬP (b): {name} ================")
    
    norms = np.linalg.norm(embeddings_matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    norm_embeddings = embeddings_matrix / norms
    
    user_embs = build_user_embedding_matrix(norm_embeddings, user_history, id_to_index)

    start_time = time.time()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        
    def scores_fn(ub_tensor):
        indices = ub_tensor.tolist() if isinstance(ub_tensor, torch.Tensor) else list(ub_tensor)
        batch_users = [int_to_user[int(idx)] for idx in indices]
        batch_vecs = np.array([user_embs.get(u, np.zeros(norm_embeddings.shape[1])) for u in batch_users])
        scores = np.dot(batch_vecs, norm_embeddings.T)
        return torch.tensor(scores, dtype=torch.float32)

    # Gọi hàm evaluate với cand là cand_tensor_b (Toàn bộ item)
    eval_results_b = evaluate(
        scores_fn=scores_fn, 
        target=target_dict, 
        mask=mask_dict, 
        ks=(5, 10, 20), 
        cand=cand_tensor_b,
        bs=1024, 
        tie_seed=0
    )
    end_time = time.time()
    total_time = end_time - start_time
    
    # Thống kê phần cứng thực nghiệm theo yêu cầu
    vram_gb = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
    latency_ms_per_user = (total_time * 1000) / max(1, len(target_dict))
    
    print(f"--- THỐNG KÊ HIỆU NĂNG ({name}) ---")
    print(f"Thời gian chạy (s/epoch): {total_time:.4f} s")
    print(f"Bộ nhớ VRAM (GB): {vram_gb:.4f} GB")
    print(f"Độ trễ (ms/user): {latency_ms_per_user:.4f} ms/user")
    
    print(f"\n--- CHỈ SỐ ĐÁNH GIÁ CHI TIẾT ---")
    for metric_name, values in eval_results_b.items():
        print(f"{metric_name}: {values.mean():.4f}")
        
    return eval_results_b

Số lượng khóa được phép gợi ý ở tập (b): 3769 / 3769


In [7]:
# Biến thể 1: Content-KNN sử dụng Văn bản Gốc (Raw Text Embeddings)
results_raw_b = evaluate_content_knn_model_on_b(raw_embeddings, name="Content-KNN (Văn bản gốc - Tập b)")

# Biến thể 2: Content-KNN sử dụng Hồ sơ LLM (LLM Profile Embeddings)
results_raw_b = evaluate_content_knn_model_on_b(llm_embeddings, name="Content-KNN (Hồ sơ LLM - Tập b)")


================ ĐANG CHẠY ĐÁNH GIÁ TẬP (b): Content-KNN (Văn bản gốc - Tập b) ================
--- THỐNG KÊ HIỆU NĂNG (Content-KNN (Văn bản gốc - Tập b)) ---
Thời gian chạy (s/epoch): 1.7103 s
Bộ nhớ VRAM (GB): 0.0000 GB
Độ trễ (ms/user): 0.5121 ms/user

--- CHỈ SỐ ĐÁNH GIÁ CHI TIẾT ---
Recall@5: 0.0307
Recall@10: 0.0715
Recall@20: 0.0948
NDCG@5: 0.0200
NDCG@10: 0.0326
NDCG@20: 0.0386
HR@5: 0.0326
HR@10: 0.0754
HR@20: 0.0991

================ ĐANG CHẠY ĐÁNH GIÁ TẬP (b): Content-KNN (Hồ sơ LLM - Tập b) ================
--- THỐNG KÊ HIỆU NĂNG (Content-KNN (Hồ sơ LLM - Tập b)) ---
Thời gian chạy (s/epoch): 1.6683 s
Bộ nhớ VRAM (GB): 0.0000 GB
Độ trễ (ms/user): 0.4995 ms/user

--- CHỈ SỐ ĐÁNH GIÁ CHI TIẾT ---
Recall@5: 0.0236
Recall@10: 0.0333
Recall@20: 0.0541
NDCG@5: 0.0156
NDCG@10: 0.0189
NDCG@20: 0.0243
HR@5: 0.0257
HR@10: 0.0362
HR@20: 0.0596
